# 02 — Image inventory

For each dry-season-year, group candidate scenes by sensor-date with AOI cloud % and coverage %, then select the single clearest full-coverage image (or flag a gap year).

In [ ]:
import os
REPO_DIR = '/content/Shoreline-Dynamics'
if os.path.isdir(REPO_DIR):
    # Already cloned this session: pull latest main so code fixes are picked
    # up (a plain clone-if-missing would keep stale code on disk).
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone https://github.com/SKPrince1911/Shoreline-Dynamics.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q earthengine-api geemap

In [ ]:
import ee, geemap
ee.Authenticate()
ee.Initialize(project='shoreline-analysis')
print('GEE initialized:', ee.String('ok').getInfo())

In [ ]:
import sys; sys.path.append('.')
import importlib
import pandas as pd
import src.config, src.data
# Reload in case an older copy was imported earlier this session (config
# first, since data imports it).
importlib.reload(src.config)
importlib.reload(src.data)
from src import config, data

aoi = ee.Geometry.Polygon(config.aoi_coordinates())

# Retrieval lives in src/data.py so fixes propagate via `git pull` (the
# notebook cells you run are a separate copy from the repo files). Both
# helpers page results in small key-chunks to avoid Earth Engine's
# 'Too many concurrent aggregations' (HTTP 429) on high-sensor years.
def candidates_df(year):
    """Per sensor-date candidates for a dry-season-year, sorted by AOI cloud %."""
    return data.candidates_dataframe(year, aoi)

def best_df(year):
    """The single selected image (or gap record) for a dry-season-year."""
    feat = data.select_best(year, aoi)  # ee.Feature of plain scalars
    return pd.DataFrame([feat.getInfo().get('properties', {})])

## Test year 1995

In [ ]:
candidates_df(1995)

In [ ]:
best_df(1995)

## Test year 2010

In [ ]:
candidates_df(2010)

In [ ]:
best_df(2010)

## Test year 2022 (high-sensor year: L7 + L8 + L9 + S2)

In [ ]:
candidates_df(2022)

In [ ]:
best_df(2022)

## Human review (reproducible)

Inspect each year's ranked candidates and the rank-0 true-color mosaic, then record an approved choice to a CSV so the selection is reproducible and auditable. Approvals override the automatic `select_best` pick.

In [ ]:
import json, datetime

APPROVALS_PATH = 'outputs/inventory_approvals.csv'
APPROVAL_COLUMNS = [
    'dry_year', 'date', 'sensor', 'chosen_rank', 'aoi_cloud_pct',
    'aoi_coverage_pct', 'relaxed_coverage', 'n_scenes', 'image_ids',
    'note', 'reviewed_at_utc',
]

# Load prior approvals if present, else start an empty ledger.
if os.path.exists(APPROVALS_PATH):
    approvals = pd.read_csv(APPROVALS_PATH)
else:
    approvals = pd.DataFrame(columns=APPROVAL_COLUMNS)
approvals

In [ ]:
from IPython.display import display

def ranked_df(year):
    """All candidates for a year as a DataFrame, ordered by rank (0 = best)."""
    fc = data.year_candidates_ranked(year, aoi)
    rows = [f['properties'] for f in fc.getInfo()['features']]
    cols = ['rank', 'date', 'sensor', 'aoi_cloud_pct', 'aoi_coverage_pct',
            'n_scenes', 'slc_off', 'qualifies_95', 'qualifies_90', 'image_ids']
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df[cols].sort_values('rank').reset_index(drop=True)
    return df

def view_candidate(year, rank=0):
    """Print a candidate summary and show its true-color mosaic on a map."""
    fc = data.year_candidates_ranked(year, aoi)
    rows = [f['properties'] for f in fc.getInfo()['features']]
    row = next(r for r in rows if r['rank'] == rank)
    print(f"dry_year {year} | rank {rank} of {len(rows)} candidates: "
          f"{row['date']} {row['sensor']} | cloud {row['aoi_cloud_pct']:.2f}% | "
          f"coverage {row['aoi_coverage_pct']:.2f}%")
    img = data.candidate_mosaic_truecolor(year, aoi, rank)
    m = geemap.Map()
    m.add_basemap('SATELLITE')
    m.centerObject(aoi, 11)
    m.addLayer(img, {}, f'{year} rank {rank} true color')
    aoi_fc = ee.FeatureCollection([ee.Feature(aoi)])
    m.addLayer(aoi_fc.style(color='red', fillColor='00000000', width=2), {}, 'AOI')
    with open('data/tidal_channels.geojson') as _f:
        channels = geemap.geojson_to_ee(json.load(_f))
    m.addLayer(channels.style(color='cyan', width=2), {}, 'tidal channels')
    return m

def review(year):
    """Show the ranked candidate table, then the rank-0 candidate on a map."""
    display(ranked_df(year))
    return view_candidate(year, 0)

In [ ]:
def approve(year, rank=0, note=''):
    """Record the reviewer's chosen candidate for a year in the approvals CSV."""
    global approvals
    fc = data.year_candidates_ranked(year, aoi)
    rows = [f['properties'] for f in fc.getInfo()['features']]
    row = next(r for r in rows if r['rank'] == rank)
    entry = {
        'dry_year': year,
        'date': row['date'],
        'sensor': row['sensor'],
        'chosen_rank': rank,
        'aoi_cloud_pct': row['aoi_cloud_pct'],
        'aoi_coverage_pct': row['aoi_coverage_pct'],
        'relaxed_coverage': bool(row['qualifies_90'] and not row['qualifies_95']),
        'n_scenes': row['n_scenes'],
        'image_ids': row['image_ids'],
        'note': note,
        'reviewed_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    }
    # Replace any existing row for this year (keyed by dry_year).
    approvals = approvals[approvals['dry_year'] != year]
    approvals = pd.concat(
        [approvals, pd.DataFrame([entry], columns=APPROVAL_COLUMNS)],
        ignore_index=True,
    )
    os.makedirs(os.path.dirname(APPROVALS_PATH), exist_ok=True)
    approvals.to_csv(APPROVALS_PATH, index=False)
    print(f"Approved dry_year {year}: rank {rank} -> {row['date']} {row['sensor']} "
          f"(cloud {row['aoi_cloud_pct']:.2f}%, coverage {row['aoi_coverage_pct']:.2f}%). "
          f"Saved to {APPROVALS_PATH}")

def show_approvals():
    """Return the approvals ledger sorted by dry_year."""
    if approvals.empty:
        print('No approvals recorded yet.')
        return approvals
    return approvals.sort_values('dry_year').reset_index(drop=True)

def effective_selection(year):
    """Approved row (review_status='approved') if present, else select_best ('auto')."""
    match = approvals[approvals['dry_year'] == year]
    if not match.empty:
        row = match.iloc[0].to_dict()
        row['review_status'] = 'approved'
        return row
    row = data.select_best(year, aoi).getInfo().get('properties', {})
    row['review_status'] = 'auto'
    return row

### Demonstration (1995, 2010, 2022)

Review each year, peek at an alternate candidate for 1995, then approve a choice per year and show the ledger.

In [ ]:
review(1995)

In [ ]:
view_candidate(1995, 1)

In [ ]:
approve(1995, 0, note='test')

In [ ]:
review(2010)

In [ ]:
approve(2010, 0)

In [ ]:
review(2022)

In [ ]:
approve(2022, 0)

In [ ]:
show_approvals()